# DTL: Thematic validation

This script conducts thematic validation of the LULC map generated in Step 4 using independent reference validation points

## 1. Setup and prepare validation data

Initialize the Earth Engine environment and load the final LULC map and independent reference validation points. The validation is currently performed for the first province and can subsequently be extended to all provinces

In [ ]:
# A. Setup

# EE Initialization
# ------------------------------------------------------------
!python -m pip install .. --quiet

import ee
import numpy as np
import pandas as pd

from luma_ge.accuracy import (
    thematic_accuracy,
    sample_size_calculator,
    validation_error_flag,
)

ee.Authenticate()
ee.Initialize(project='epistem-lumastack')

VERSION = 'v6'



# List Classification Classes
# ------------------------------------------------------------
CLASS_NAMES = {
    1:  'Primary Dryland Forest',
    2:  'Secondary Dryland Forest',
    3:  'Primary Mangrove Forest',
    4:  'Secondary Mangrove Forest',
    5:  'Primary Swamp Forest',
    6:  'Secondary Swamp Forest',
    7:  'Plantation Forest',
    8:  'Rubber Monoculture',
    9:  'Oil palm Monoculture',
    10: 'Cacao Monoculture',
    11: 'Coconut monoculture',
    12: 'Other Monoculture',
    13: 'Other Cropland',
    14: 'Coffee agroforestry',
    15: 'Rubber agroforestry',
    16: 'Mixed/home garden',
    17: 'Paddy field',
    18: 'Grass or Savanna',
    19: 'Shrub',
    20: 'Settlement',
    21: 'Cleared Land',
    22: 'Mining area',
    23: 'Waterbody',
    24: 'Fish pond',
}

CLASS_IDS = list(CLASS_NAMES.keys())

VALIDATION_CLASS_PROPERTY = 'IDe'



# Call AOI
# ------------------------------------------------------------
aoi = ee.FeatureCollection(
    'projects/epistem2/assets/AOI_Sumatra'
).geometry()



# Call Validation Points
# ------------------------------------------------------------

validation_fc = ee.FeatureCollection(
    'projects/epistem2/assets/Sumatra_Validation_Points'
).filterBounds(aoi)



# Call Final LULC Map
# ------------------------------------------------------------
final_map = ee.Image(
    f'projects/epistem2/assets/final_lulc_stack_Aceh_2020_{VERSION}'
)

print("✓ Part 5 initialized")
print("Validation property:", VALIDATION_CLASS_PROPERTY)
print("Number of validation points:", validation_fc.size().getInfo())
print("Final map bands:", final_map.bandNames().getInfo())

In [ ]:
# B. Validation Data QA

print("============================================================")
print("VALIDATION DATA QA")
print("============================================================")

total_validation = validation_fc.size().getInfo()

reference_hist = (
    validation_fc
    .aggregate_histogram(VALIDATION_CLASS_PROPERTY)
    .getInfo()
)

reference_hist = {
    int(k): int(v)
    for k, v in reference_hist.items()
}

print(f"\nTotal validation points: {total_validation}")

print("\nReference class distribution:")
print("-" * 65)
print(f"{'ID':>4}  {'Class':<32} {'N':>8} {'Share':>8}")
print("-" * 65)

for class_id in CLASS_IDS:
    n = reference_hist.get(class_id, 0)
    share = (n / total_validation * 100) if total_validation else 0

    print(
        f"{class_id:>4}  "
        f"{CLASS_NAMES[class_id]:<32} "
        f"{n:>8} "
        f"{share:>7.1f}%"
    )

print("-" * 65)

missing_reference_classes = [
    class_id
    for class_id in CLASS_IDS
    if reference_hist.get(class_id, 0) == 0
]

print(
    "\nClasses without validation samples:",
    missing_reference_classes
)

print(
    "Number of classes without validation samples:",
    len(missing_reference_classes)
)

In [ ]:
# C. Map Class Inventory

print("============================================================")
print("MAP CLASS INVENTORY")
print("============================================================")

# Final map histogram
# ------------------------------------------------------------
map_hist = (
    final_map
    .select('classification')
    .reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=aoi,
        scale=100,
        maxPixels=1e13
    )
    .getInfo()
)

map_hist = map_hist.get('classification', {})

map_hist = {
    int(k): float(v)
    for k, v in map_hist.items()
}

print("\nClasses present in final map:")
print("-" * 65)
print(f"{'ID':>4}  {'Class':<32} {'Pixel count':>15}")
print("-" * 65)

for class_id in sorted(map_hist):
    class_name = CLASS_NAMES.get(
        class_id,
        f"Unknown class {class_id}"
    )

    print(
        f"{class_id:>4}  "
        f"{class_name:<32} "
        f"{map_hist[class_id]:>15,.0f}"
    )

print("-" * 65)

missing_map_classes = [
    class_id
    for class_id in CLASS_IDS
    if class_id not in map_hist
]

print(
    "\nTarget classes absent from final map:",
    missing_map_classes
)

In [ ]:
# D. Sample Final Map at Validation Points

sampled_validation = final_map.sampleRegions(
    collection=validation_fc,
    properties=[
        VALIDATION_CLASS_PROPERTY,
        'LULC20e',
        'IDe'
    ],
    scale=100,
    geometries=True
)

total_sampled = sampled_validation.size().getInfo()

print("============================================================")
print("VALIDATION COVERAGE")
print("============================================================")

print(f"Original validation points : {total_validation}")
print(f"Usable map samples         : {total_sampled}")

coverage = (
    total_sampled / total_validation * 100
    if total_validation
    else 0
)

print(f"Validation coverage        : {coverage:.2f}%")
print(f"Excluded / NoData points   : {total_validation - total_sampled}")

## 2. Inspect reference data

Check the number of reference points, available attributes, and class distribution before performing the accuracy assessment. The numerical `IDe` field is used as the reference class identifier because it corresponds to the class IDs used by the LULC map

In [ ]:
# E. Prediction Distribution at Validation Points

prediction_hist = (
    sampled_validation
    .aggregate_histogram('classification')
    .getInfo()
)

prediction_hist = {
    int(k): int(v)
    for k, v in prediction_hist.items()
}

print("============================================================")
print("PREDICTION DISTRIBUTION AT VALIDATION POINTS")
print("============================================================")

print("-" * 65)
print(f"{'ID':>4}  {'Class':<32} {'N':>8} {'Share':>8}")
print("-" * 65)

for class_id in sorted(prediction_hist):

    class_name = CLASS_NAMES.get(
        class_id,
        f"Unknown class {class_id}"
    )

    n = prediction_hist[class_id]

    share = (
        n / total_sampled * 100
        if total_sampled
        else 0
    )

    print(
        f"{class_id:>4}  "
        f"{class_name:<32} "
        f"{n:>8} "
        f"{share:>7.1f}%"
    )

print("-" * 65)

In [ ]:
# F. Correctness Flag

def add_correctness(feature):
    reference = ee.Number(
        feature.get(VALIDATION_CLASS_PROPERTY)
    )

    predicted = ee.Number(
        feature.get('classification')
    )

    return feature.set(
        'correct',
        predicted.eq(reference)
    )


validation_with_accuracy = sampled_validation.map(
    add_correctness
)

correct_count = (
    validation_with_accuracy
    .filter(ee.Filter.eq('correct', True))
    .size()
    .getInfo()
)

incorrect_count = (
    validation_with_accuracy
    .filter(ee.Filter.eq('correct', False))
    .size()
    .getInfo()
)

print("============================================================")
print("POINT-LEVEL CORRECTNESS")
print("============================================================")

print("Correct predictions  :", correct_count)
print("Incorrect predictions:", incorrect_count)

if total_sampled:
    print(
        "Observed accuracy    : "
        f"{correct_count / total_sampled * 100:.2f}%"
    )

## 3. Extract map predictions at validation points

Sample the final LULC map at the locations of the independent reference points. The `classification` and `confidence` bands are extracted for each point. Points with masked or unavailable map predictions are excluded from the subsequent accuracy assessment

In [ ]:
# G. Thematic Accuracy Assessment

accuracy_assessor = thematic_accuracy()

success, results = accuracy_assessor.run_accuracy_assessment(
    lcmap=final_map,
    validation_data=validation_fc,
    class_property=VALIDATION_CLASS_PROPERTY,
    scale=100,
    confidence=0.95,
)

if not success:
    raise RuntimeError(
        f"Assessment failed: "
        f"{results.get('error', 'unknown error')}"
    )

summary = accuracy_assessor.format_accuracy_summary(
    results
)

print("============================================================")
print("THEMATIC ACCURACY SUMMARY")
print("============================================================")

for name, value in summary.items():
    print(f"{name}: {value}")

In [ ]:
# H. Class Level Accuracy Table

producer = results['producer_accuracy']
user = results['user_accuracy']
f1 = results['f1_scores']

rows = []

for class_id in CLASS_IDS:

    # Library output appears to use class IDs as positions
    # Adjust only if inspection shows a different ordering
    idx = class_id

    pa = producer[idx] if idx < len(producer) else 0
    ua = user[idx] if idx < len(user) else 0
    f1_value = f1[idx] if idx < len(f1) else 0

    n_reference = reference_hist.get(class_id, 0)

    rows.append({
        'class_id': class_id,
        'class_name': CLASS_NAMES[class_id],
        'reference_n': n_reference,
        'producer_accuracy': pa,
        'user_accuracy': ua,
        'f1_score': f1_value,
        'omission_error': 1 - pa,
        'commission_error': 1 - ua,
    })

accuracy_df = pd.DataFrame(rows)

print(accuracy_df.to_string(index=False))

## 4. Run thematic accuracy assessment

Thematic accuracy is assessed using the `luma_ge` accuracy module. The assessment provides overall accuracy, Cohen's kappa, producer's accuracy, user's accuracy, F1-score, confidence interval, sample size, and the confusion matrix

In [ ]:
# I. Confusion Matrix

confusion = np.array(
    results['confusion_matrix']
)

confusion_df = pd.DataFrame(
    confusion,
    index=[f"Ref_{i}" for i in range(confusion.shape[0])],
    columns=[f"Pred_{i}" for i in range(confusion.shape[1])]
)

print("============================================================")
print("CONFUSION MATRIX")
print("============================================================")

print(confusion_df)

In [ ]:
# J. Confidence vs Correctness

confidence_stats = (
    validation_with_accuracy
    .aggregate_stats('confidence')
    .getInfo()
)

print("============================================================")
print("CONFIDENCE STATISTICS")
print("============================================================")

print(confidence_stats)

# Confidence bins
# ------------------------------------------------------------
confidence_bins = [
    (0, 20),
    (20, 40),
    (40, 60),
    (60, 80),
    (80, 100),
]

print("\nConfidence vs correctness")
print("-" * 60)

for lower, upper in confidence_bins:

    subset = validation_with_accuracy.filter(
        ee.Filter.And(
            ee.Filter.gte('confidence', lower),
            ee.Filter.lt('confidence', upper + 1)
        )
    )

    n = subset.size().getInfo()

    correct = (
        subset
        .filter(ee.Filter.eq('correct', True))
        .size()
        .getInfo()
    )

    accuracy = (
        correct / n * 100
        if n > 0
        else np.nan
    )

    print(
        f"{lower:>3}-{upper:<3} : "
        f"N={n:>4}, "
        f"Correct={correct:>4}, "
        f"Accuracy={accuracy:>6.2f}%"
    )

## 5. Analyse classification errors

Examine the confusion matrix and reference-to-prediction relationships to identify the dominant class confusions and systematic classification errors

In [ ]:
# K. Reference vs Prediction Cross Tabulation

def make_reference_prediction_pair(feature):

    reference = ee.Number(
        feature.get(VALIDATION_CLASS_PROPERTY)
    )

    predicted = ee.Number(
        feature.get('classification')
    )

    pair = (
        reference.format('%d')
        .cat(' -> ')
        .cat(predicted.format('%d'))
    )

    return feature.set(
        'reference_prediction',
        pair
    )


validation_pairs = sampled_validation.map(
    make_reference_prediction_pair
)


pair_hist = (
    validation_pairs
    .aggregate_histogram('reference_prediction')
    .getInfo()
)


print("============================================================")
print("REFERENCE → PREDICTION CROSS-TABULATION")
print("============================================================")

print(f"{'Reference → Prediction':<30}{'Count':>8}")
print("-" * 40)

for pair, count in sorted(pair_hist.items()):
    print(f"{pair:<30}{count:>8}")

print("-" * 40)

## 6. Export validation results

Export the point-level validation results both as an Earth Engine asset and as a CSV file for further spatial and statistical analysis

In [ ]:
# L. Export Validation Results to GEE Asset

task = ee.batch.Export.table.toAsset(
    collection=validation_with_accuracy,
    description=f'Sumatra_LULC_validation_{VERSION}',
    assetId=f'projects/epistem2/assets/Sumatra_LULC_validation_{VERSION}'
)

task.start()

print("✓ Export task started")
print("Task ID:", task.id)

In [ ]:
# M. Export validation results to Google Drive

task_drive = ee.batch.Export.table.toDrive(
    collection=validation_with_accuracy,
    description=f'Sumatra_LULC_validation_{VERSION}',
    folder='GEE_Validation',
    fileNamePrefix=f'Sumatra_LULC_validation_{VERSION}',
    fileFormat='CSV'
)

task_drive.start()

print("✓ Google Drive export task started")
print("Task ID:", task_drive.id)

## 7. Validation summary

Review the overall and class-level accuracy, confusion patterns, sample distribution, and confidence of the validation results. The results should be interpreted together with the spatial distribution and quality of the reference samples before drawing conclusions about the performance of the LULC map

In [ ]:
# N. Final validation report

print("\n")
print("=" * 70)
print("LULC VALIDATION REPORT")
print("=" * 70)

print(f"Map version             : {VERSION}")
print(f"Reference dataset       : Sumatra_Validation_Points")
print(f"Reference class field   : {VALIDATION_CLASS_PROPERTY}")
print(f"Map scale               : 100 m")

print("\nValidation:")
print(f"  Total reference points : {total_validation}")
print(f"  Valid map samples      : {total_sampled}")
print(f"  Excluded / NoData      : {total_validation - total_sampled}")
print(f"  Coverage               : {coverage:.2f}%")

print("\nAccuracy:")
print(f"  Overall accuracy       : {summary.get('overall_accuracy')}")
print(f"  Kappa                  : {summary.get('kappa')}")
print(f"  Confidence interval    : {summary.get('confidence_interval')}")
print(f"  Sample size            : {summary.get('sample_size')}")

print("\nClass inventory:")
print(f"  Target classes         : {len(CLASS_IDS)}")
print(f"  Reference classes      : {len(reference_hist)}")
print(f"  Map classes            : {len(map_hist)}")
print(
    f"  Missing from reference: "
    f"{len(missing_reference_classes)}"
)
print(
    f"  Missing from map      : "
    f"{len(missing_map_classes)}"
)

print("=" * 70)